In [1]:
!nvidia-smi
!nvcc --version

Tue Jun  9 08:46:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:06:00.0 Off |                  Off |
|  0%   26C    P8             23W /  450W |       1MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!python --version 

Python 3.12.3


In [1]:
!pip install torch==2.7+cu126 torchvision==0.22.0+cu126
!pip install diffusers==0.32.1 transformers==4.38.2 accelerate==0.27.2
!pip install torchsummary==1.5.1 lpips==0.1.4
!pip install numpy==2.1.3 scikit-learn==1.8.0
!pip install matplotlib==3.10.9 seaborn==0.13.2 pandas==3.0.3
!pip install Pillow==12.2.0 tqdm==4.67.3

ERROR: Could not find a version that satisfies the requirement torch==2.7+cu126 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0)
ERROR: No matching distribution found for torch==2.7+cu126


In [2]:
!pip install hf_transfer --break-system-packages

In [5]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torchsummary import summary

from PIL import Image

import torch.nn.functional as F
from tqdm import tqdm

from diffusers.models import AutoencoderKL
from diffusers import AutoencoderKL

from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix, roc_curve, roc_auc_score, RocCurveDisplay
import seaborn as sns

import lpips

from torchvision.models import resnet50, ResNet50_Weights

from torchvision.models.feature_extraction import create_feature_extractor
import torch.nn as nn
'''
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0

import keras_hub
'''

'\nimport tensorflow as tf\nfrom tensorflow import keras\nfrom tensorflow.keras import layers\nfrom tensorflow.keras.applications import EfficientNetB0\n\nimport keras_hub\n'

In [ ]:
"""
check_versions.py
Muestra la versión de cada librería del proyecto. No instala nada.
"""

import sys
import subprocess

GREEN  = "\033[92m"
RED    = "\033[91m"
YELLOW = "\033[93m"
CYAN   = "\033[96m"
BOLD   = "\033[1m"
RESET  = "\033[0m"

def ok(v):  return f"{GREEN}✔  {v}{RESET}"
def err(e): return f"{RED}✘  {e}{RESET}"

print(f"\n{BOLD}{CYAN}{'─'*50}")
print("  Python & Sistema")
print(f"{'─'*50}{RESET}")
print(f"  {'Python':<18}: {ok(sys.version.split()[0])}")

try:
    out = subprocess.check_output(["nvcc", "--version"], stderr=subprocess.DEVNULL).decode()
    line = next((l for l in out.splitlines() if "release" in l), "desconocida")
    print(f"  {'CUDA (nvcc)':<18}: {ok(line.strip())}")
except Exception:
    print(f"  {'CUDA (nvcc)':<18}: {YELLOW}⚠  nvcc no encontrado{RESET}")

LIBS = [
    ("numpy",       "numpy"),
    ("matplotlib",  "matplotlib"),
    ("pandas",      "pandas"),
    ("torch",       "torch"),
    ("torchvision", "torchvision"),
    ("torchsummary","torchsummary"),
    ("PIL",         "Pillow"),
    ("tqdm",        "tqdm"),
    ("diffusers",   "diffusers"),
    ("sklearn",     "scikit-learn"),
    ("seaborn",     "seaborn"),
    ("lpips",       "lpips"),
]

print(f"\n{BOLD}{CYAN}{'─'*50}")
print("  Versiones de librerías")
print(f"{'─'*50}{RESET}")

for import_name, display_name in LIBS:
    try:
        mod = __import__(import_name)
        version = getattr(mod, "__version__", "sin __version__")
        print(f"  {display_name:<18}: {ok(version)}")
    except ImportError as e:
        print(f"  {display_name:<18}: {err(e)}")

# ── Soporte GPU por librería ──────────────────────────────────────────────────
print(f"\n{BOLD}{CYAN}{'─'*50}")
print("  Soporte GPU por librería")
print(f"{'─'*50}{RESET}")

# torch
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram     = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"  {'torch':<18}: {ok(f'GPU  ·  {gpu_name}  ·  {vram:.1f} GB VRAM')}")
        print(f"  {'  CUDA':<18}: {ok(torch.version.cuda)}")
        print(f"  {'  cuDNN':<18}: {ok(torch.backends.cudnn.version())}")
        n = torch.cuda.device_count()
        if n > 1:
            print(f"  {'  GPUs totales':<18}: {ok(n)}")
    else:
        print(f"  {'torch':<18}: {YELLOW}⚠  solo CPU{RESET}")
except ImportError:
    print(f"  {'torch':<18}: {RED}✘  no instalado{RESET}")

# jax (opcional, puede estar presente en entornos ML)
try:
    import jax
    devices = jax.devices()
    gpu_devs = [d for d in devices if "gpu" in str(d).lower()]
    if gpu_devs:
        print(f"  {'jax':<18}: {ok(f'GPU  ·  {gpu_devs}')}")
    else:
        print(f"  {'jax':<18}: {YELLOW}⚠  solo CPU{RESET}")
except ImportError:
    pass  # jax no es parte del proyecto, se omite si no está

print(f"\n{BOLD}{'─'*50}{RESET}\n")


──────────────────────────────────────────────────
  Python & Sistema
──────────────────────────────────────────────────
  Python            : ✔  3.12.3
  CUDA (nvcc)       : ✔  Cuda compilation tools, release 12.8, V12.8.93

──────────────────────────────────────────────────
  Versiones de librerías
──────────────────────────────────────────────────
  numpy             : ✔  2.1.3
  matplotlib        : ✔  3.10.9
  pandas            : ✔  3.0.3
  torch             : ✔  2.8.0+cu128
  torchvision       : ✔  0.23.0+cu128
  torchsummary      : ✔  sin __version__
  Pillow            : ✔  12.2.0
  tqdm              : ✔  4.67.3
  diffusers         : ✔  0.32.1
  scikit-learn      : ✔  1.8.0
  seaborn           : ✔  0.13.2
  lpips             : ✔  sin __version__

──────────────────────────────────────────────────
  Soporte GPU por librería
──────────────────────────────────────────────────
  torch             : ✔  GPU  ·  NVIDIA GeForce RTX 4090  ·  23.5 GB VRAM
    CUDA            : ✔  12.8
  